# Week 1 - Churn Prediction Pipeline
## Rigour Pass: Leakage-Free Evaluation
**Goal:** re-run the evaluation without leakage, choose the threshold properly, calibrate, and rebuild the segments.

**Author:** Martin James | github.com/M20Jay

## Section 1 - Import Libraries and Load Data
We import the tools and load the Telco churn CSV directly, so the notebook runs from a clean clone.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score,StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import precision_recall_curve



df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)

(7043, 21)


## Section 2 - Clean Data and Engineer Features
We fix TotalCharges, encode the target, and create 3 features: tenure_group, charges_per_month, is_high_value.

In [15]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df["tenure_group"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 72], labels=["New", "Mid", "Loyal"]).astype(str)
df["charges_per_month"] = df["TotalCharges"] / (df["tenure"] + 1)
df["is_high_value"] = (df["MonthlyCharges"] > 70).astype(int)

## Section 3 - Train/Test Split
We split off the test set FIRST. It is touched once, at the very end.

In [16]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(5634, 22) (1409, 22)
0.265 0.265


## Section 4 - Build the Pipeline
One object holds encoding, scaling, SMOTE and the model, so training and serving cannot drift apart.

In [17]:

cat_cols = X_train.select_dtypes(include= "object").columns.tolist()
num_cols = X_train.select_dtypes(exclude="object").columns.tolist()


def make(model):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", StandardScaler(), num_cols),

    ])

    return Pipeline([("prep", prep), ("bal", SMOTE(random_state=42)), ("m", model)])


models = {
    "LogReg" : LogisticRegression(max_iter=1000, random_state=42),
    "RandForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss"),
}

cv= StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Section 5 - Leakage-Free Cross-Validation
SMOTE runs inside each fold, on the training part only. Scored by ROC-AUC and PR-AUC.

In [18]:
for metric in ["roc_auc", "average_precision"]:
    print(metric)
    for name, model in models.items():
        s = cross_val_score(make(model), X_train, y_train, cv=cv, scoring=metric)
        print(f"  {name:10s} {s.mean():.3f} ± {s.std():.3f}")

roc_auc
  LogReg     0.847 ± 0.012
  RandForest 0.819 ± 0.010
  XGBoost    0.823 ± 0.008
average_precision
  LogReg     0.666 ± 0.017
  RandForest 0.599 ± 0.030
  XGBoost    0.629 ± 0.022


## Section 6 - Does SMOTE Help?
We compare SMOTE against class weights, because churn is 26.5 percent and may not need resampling.

In [19]:
def make_plain(model):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", StandardScaler(), num_cols),
    ])
    return Pipeline([("prep", prep), ("m", model)])

spw = (y_train == 0).sum() / (y_train == 1).sum()

variants = {
    "LogReg + SMOTE":   make(LogisticRegression(max_iter=1000, random_state=42)),
    "LogReg + weights": make_plain(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    "LogReg + nothing": make_plain(LogisticRegression(max_iter=1000, random_state=42)),
    "XGB + SMOTE":      make(XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss")),
    "XGB + weights":    make_plain(XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss", scale_pos_weight=spw)),
    "XGB + nothing":    make_plain(XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss")),
}

for metric in ["roc_auc", "average_precision"]:
    print(metric)
    for name, pipe in variants.items():
        s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=metric)
        print(f"  {name:18s} {s.mean():.3f} ± {s.std():.3f}")

roc_auc
  LogReg + SMOTE     0.847 ± 0.012
  LogReg + weights   0.848 ± 0.011
  LogReg + nothing   0.848 ± 0.011
  XGB + SMOTE        0.823 ± 0.008
  XGB + weights      0.822 ± 0.005
  XGB + nothing      0.824 ± 0.007
average_precision
  LogReg + SMOTE     0.666 ± 0.017
  LogReg + weights   0.668 ± 0.014
  LogReg + nothing   0.669 ± 0.015
  XGB + SMOTE        0.629 ± 0.022
  XGB + weights      0.620 ± 0.010
  XGB + nothing      0.629 ± 0.016


## Section 7 - Out-of-Fold Threshold Selection
We pick the decision threshold from honest out-of-fold predictions, not the default 0.5.

In [ ]:
best_pipe = make_plain(LogisticRegression(max_iter=1000, random_state=42))

# out-of-fold probabilities : every row predicted by a model that never saw it
oof = cross_val_predict(best_pipe,X_train, y_train, cv, method="predict_proba")[:,1]

prec, rec, thr = precision_recall_curve(y_train,oof)

SyntaxError: invalid syntax (498307173.py, line 1)

## Section 8 - Calibration
We check whether the probabilities mean what they say, and fix them with isotonic calibration.

## Section 9 - Final Evaluation on the Test Set
One look at the test set, with the chosen model and threshold.

## Section 10 - Rebuild Customer Segments
Out-of-fold churn probability for all 7,043 customers, combined with ARPU.

## Section 11 - Save the Pipeline
We save a single pipeline object for the API.

## Section 12 - Key Findings
Before and after numbers, and what changed.